In [12]:
import itertools
import os
import random
from collections import deque, namedtuple
from datetime import datetime

import gymnasium as gym
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from src.agents.common import calc_advantage, norm_advantage
from src.networks.dqn import DQN
from src.networks.mlp import MLP

In [13]:
n_episodes = 1000
LR = 1.5e-3
K_EPOCH = 4
num_tests=100
BATCH_SIZE = 32
GAMMA = .95
EPS = .2
ENTROPY_COEFF = .01

In [14]:
run_name = f"ppo_bj_lr{LR}_ne{n_episodes}_k{K_EPOCH}_{datetime.now()}"
writer = SummaryWriter(f"./logs/{run_name}")

In [15]:
env = gym.make("Blackjack-v1", sab=False)
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
env.action_space, env.observation_space

(Discrete(2), Tuple(Discrete(32), Discrete(11), Discrete(2)))

In [16]:
policy = MLP(len(env.observation_space), env.action_space.n)
critic = DQN(len(env.observation_space), 1, 20)

optimizer_policy = torch.optim.Adam(policy.net.parameters(), LR)
optimizer_critic = torch.optim.Adam(critic.net.parameters(), LR)

In [17]:
EpStep = namedtuple('episode_step', (
    'state',
    'next_state',
    'action',
    'reward',
    'done',
    'probs',
    ))

LearningStep = namedtuple("learning_step", (
    "state",
    "next_state",
    "action",
    "reward",
    "done",
    "probs",
    "advantage",
))

In [18]:
@torch.no_grad
def get_critiques(critic: torch.nn.Module, observations: torch.Tensor): 
    return critic(observations).squeeze(-1)

def evaluate_action(policy: torch.nn.Module, observations: torch.Tensor, actions: torch.Tensor):
    probs = policy(observations)
    dist = torch.distributions.Categorical(probs)
    action = dist.log_prob(actions)
    return action, dist.entropy().mean()

def update_critic(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: list[LearningStep],
    gamma=GAMMA,
    ): 
    batch = LearningStep(*zip(*batch, strict=False))

    states = torch.stack(batch.state) 
    next_states = torch.stack(batch.next_state)
    rewards = torch.stack(batch.reward)
    dones = torch.stack(batch.done)

    q_pred = model(states).squeeze(1)
    with torch.no_grad():
        q_next = model(next_states).squeeze(1)
        q_target = rewards + gamma * q_next * (1 - dones)

    loss = F.mse_loss(q_pred, q_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def update_policy(
    model: torch.nn.Module,
    critic: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: list[LearningStep],
    gamma=GAMMA,
):
    batch = LearningStep(*zip(*batch, strict=False))

    old_log_probs = torch.stack(batch.probs).detach()
    actions_t     = torch.stack(batch.action)
    states_t      = torch.stack(batch.state) 
    advantage     = torch.stack(batch.advantage) 

    new_log_probs, entropy = evaluate_action(model, states_t, actions_t)
    ratio = torch.exp(new_log_probs - old_log_probs)

    surr1 = ratio * advantage
    surr2 = torch.clamp(ratio, 1 - EPS, 1 + EPS) * advantage
    actor_loss = -torch.min(surr1, surr2).mean()
    
    loss = actor_loss - entropy * ENTROPY_COEFF

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



def get_action(policy: torch.nn.Module, observations: torch.Tensor):
    probs = policy(observations)
    dist = torch.distributions.Categorical(probs)
    action = torch.argmax(probs)
    return int(action.item()), action, dist.log_prob(action).detach()


def make_batches(data, n = BATCH_SIZE):
    new_data = list(data)
    random.shuffle(new_data)
    return itertools.batched(new_data, n, strict=False)


def compute_advantages(critic, data, gamma):
    batch = EpStep(*zip(*data, strict=False))
    rewards       = torch.stack(batch.reward)
    states_t      = torch.stack(batch.state) 
    critiques = get_critiques(critic, states_t)
    return calc_advantage(rewards, critiques, gamma)


def save_checkpoint(
    state_dict: dict,
    epoch: int,
    reward: float,
    checkpoint_dir: str = "./data/checkpoints",
    ttl_epochs: int = 100,
    best_reward_tracker: list = None,
) -> bool:
    if best_reward_tracker is None:
        best_reward_tracker = [-float("inf")]
    os.makedirs(checkpoint_dir, exist_ok=True)
    saved = False
    
    is_ttl_match = (epoch % ttl_epochs == 0) and (epoch > 0)
    is_best_reward = reward > best_reward_tracker[0]
    
    if is_best_reward:
        best_reward_tracker[0] = reward
        best_path = os.path.join(checkpoint_dir, "best_checkpoint.pt")
        
        save_payload = {
            **state_dict,
            "epoch": epoch,
            "reward": reward,
        }
        torch.save(save_payload, best_path)
        print(f"[Checkpoint] New best reward achieved ({reward:.4f}). Saved to {best_path}")
        saved = True
        
    if is_ttl_match:
        ttl_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pt")
        
        save_payload = {
            **state_dict,
            "epoch": epoch,
            "reward": reward,
        }
        torch.save(save_payload, ttl_path)
        print(f"[Checkpoint] TTL interval reached (Epoch {epoch}). Saved to {ttl_path}")
        saved = True

    return saved

# Training

In [19]:
best_reward_tracker = [float("-inf")]
len(best_reward_tracker), best_reward_tracker

(1, [-inf])

In [ ]:
for e in range(n_episodes):
    true_ep = e+1

    state, info = env.reset()
    done = False

    episode_history = deque([], maxlen=10_000)


    while not done:
        state = torch.tensor(state, dtype=torch.float32)
        action, raw_action, prob = get_action(policy, state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated


        episode_history.append(EpStep(
            state,
            torch.tensor(next_state, dtype=torch.float32),
            raw_action,
            torch.tensor(reward, dtype=torch.float32),
            torch.tensor(done, dtype=torch.int8),
            prob,
            ))
        state = next_state


    # no normalization
    advantages = compute_advantages(critic, episode_history, GAMMA)
    items = [LearningStep(*step, d) for step, d in zip(episode_history, advantages, strict=False)]

    # TRAINING PASS:
    for _ in range(K_EPOCH):
        for batch in make_batches(items, BATCH_SIZE):
            update_critic(
                critic,
                optimizer_critic,
                batch,
                )
        update_policy(policy, critic, optimizer_policy, items)

    sum_reward = (sum(step.reward for step in episode_history)).item()
    writer.add_scalar("train/reward", sum_reward, true_ep)

    state_to_save = {
        "policy_state_dict": policy.state_dict(),
        "optimizer_policy_state_dict": optimizer_policy.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "optimizer_critic_state_dict": optimizer_critic.state_dict(),
    }

    save_checkpoint(
        state_dict=state_to_save,
        epoch=true_ep,
        reward=sum_reward,
        checkpoint_dir="./data/checkpoints",
        ttl_epochs=n_episodes // 10,
        best_reward_tracker=best_reward_tracker,
    )


In [21]:
from src.utils import run_tests

test_env = gym.make("Blackjack-v1", render_mode=None)
test_env = gym.wrappers.RecordEpisodeStatistics(test_env, buffer_length=num_tests)

policy = MLP(len(env.observation_space), env.action_space.n)
checkpoint = torch.load("./data/checkpoints/best_checkpoint.pt")
policy.load_state_dict(checkpoint["policy_state_dict"])
policy.eval()

class Mock:
    def __init__(self, net):
        self.net = net

    def act(self, x):
        obs = torch.Tensor(x)
        return get_action(self.net, obs)[0]


test_agent = Mock(policy)
avg, _min, _max, win_rate = run_tests(test_agent, test_env, writer, num_tests)
print("avg: ", avg, _min, _max, f"{win_rate:.1%}", )

avg:  -0.09 -1.0 1.0 43.0%


In [ ]:
policy = MLP(len(env.observation_space), env.action_space.n)
checkpoint = torch.load("./data/checkpoints/best_checkpoint.pt")
policy.load_state_dict(checkpoint["policy_state_dict"])
policy.eval()

dummy_input = torch.randn(1, len(env.observation_space))
torch.onnx.export(
    policy,
    dummy_input,
    f"./data/latest.onnx",
    input_names=["obs"],
    output_names=["action_probs"],
    dynamic_axes={"obs": {0: "batch"}, "action_probs": {0: "batch"}},
    external_data=False,
)
print(f"./data/{run_name}.onnx")